 # 01_RLVR_Part1_AlphaCache_v6

 Generates the multi-day feature cube parquet cache (`alpha_cache_41d_1998.parquet`).

 Run in CPU or T4 GPU (~01:30:00 to process 1998–present).

In [1]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "41"
# os.environ["CACHE_START_DATE"] = "1998-01-01"
os.environ["CACHE_START_DATE"] = "2015-01-01"

In [2]:
# CELL 1: Timing Initialization
import time

start_time = time.time()

In [3]:
# CELL 2: Environment Setup & Workspace Resolution
import os
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v3")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [4]:
# CELL 3: Load Data & Initialize Screener
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from core.settings import TradingConfig, CacheConfig
from data_pipeline.cache import CheckpointAlphaCache
from data_pipeline.loader import load_processed_data
from data_pipeline.screener import UniverseScreener
from data_pipeline.utils import get_master_trading_calendar

print("Loading preprocessed data...")
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df

# Tripwire validation on processed feature schemas (Gen 13 Contract)
expected_macro_cols = [
    "Macro_Trend",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "Mkt_Vol_63d_Z",
]
rsi_col = "RSI_14" if "RSI_14" in features_df.columns else "RSI"
atrp_col = "Vol_ATRP_20" if "Vol_ATRP_20" in features_df.columns else "ATRP"
expected_feature_cols = ["Mom_252_21", rsi_col, atrp_col]

missing_macro = set(expected_macro_cols) - set(macro_df.columns)
assert (
    not missing_macro
), f"macro_df missing expected engineered columns: {missing_macro}. Found: {macro_df.columns.tolist()}"

missing_features = set(expected_feature_cols) - set(features_df.columns)
assert (
    not missing_features
), f"features_df missing expected columns: {missing_features}. Found: {features_df.columns.tolist()}"

print("✅ Data validation passed: Schemas and contracts verified.")

config = TradingConfig()

# Reconstruct wide df_close dynamically and sync master calendar
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/output
TRADES_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/trades

Loading preprocessed data...
✅ Data validation passed: Schemas and contracts verified.


In [5]:
# CELL 4: Prepare File Paths & Directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory verified: {OUTPUT_DIR}")
print(f"Local data directory verified: {LOCAL_DATA_DIR}")

Output directory verified: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/output
Local data directory verified: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/data


In [ ]:
# CELL 5: Build Feature Cube Cache
cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()

if cache_file_path.exists():
    print(
        f"Warning: Cache file already exists at {cache_file_path}. Resuming build process."
    )

cache = CheckpointAlphaCache(
    screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK]
)

cache.build(
    start_date=CacheConfig.START_DATE,
    end_date=None,
    checkpoint_path=cache_file_path,
    save_every_n_days=2000,
)

print(f"Cache saved successfully to: {cache_file_path}")
print(f"File verified on disk: {cache_file_path.exists()}")

[INFO] Building AlphaCache with automatic resumption support:
  - Start Date:               2015-01-01
  - End Date Limit:           None (Processing all available dates)
  - Total target dates:       2950
  - Already completed:        0
  - Remaining to compute:     2950
  - Checkpoint save interval: Every 2000 processed days
  - Output location:          /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3/data/alpha_cache_41d_2015.parquet
  Processed 20/2950 remaining days... (est. 32:39 till completion)
  Processed 40/2950 remaining days... (est. 32:35 till completion)
  Processed 60/2950 remaining days... (est. 31:57 till completion)
  Processed 80/2950 remaining days... (est. 31:38 till completion)
  Processed 100/2950 remaining days... (est. 31:25 till completion)
  Processed 120/2950 remaining days... (est. 31:11 till completion)
  Processed 140/2950 remaining days... (est. 31:02 till completion)
  Processed 160/2950 remaining days... (est

In [ ]:
# CELL 6: Execution Timing
end_time = time.time()
elapsed = time.strftime("%H:%M:%S", time.gmtime(end_time - start_time))
print(f"Total Processing Time: {elapsed}")

In [ ]:
# CELL 7: Runtime Cleanup
if RUNNING_IN_COLAB:
    from google.colab import runtime

    # Disconnect instance to conserve GPU compute units
    runtime.unassign()
    print("Colab runtime unassigned. Session disconnected successfully.")